# Случайный лес: устойчивый прогноз по табличным данным

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Случайный лес».

## Что делает метод

Случайный лес позволяет предсказывать ответ (тип объекта, класс пациента, объём показателя) по таблице измерений. Метод строит множество решающих деревьев на различных подвыборках данных и усредняет их ответы, поэтому случайные ошибки отдельных деревьев взаимно гасятся.

Метод применим, когда у исследователя есть таблица: строки — наблюдения, столбцы — измеренные признаки и известный ответ. Типичные научные задачи: классификация образцов по спектральным или морфологическим признакам, оценка риска по клиническим показателям, прогноз отклика по составу. Дополнительно метод даёт оценку относительной важности признаков — какие измерения сильнее всего влияют на ответ.

В этом блокноте мы обучим случайный лес на демонстрационных данных, оценим качество и разберём, как читать результат. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вам нужно определить вид растения по набору измерений: длина листа, ширина, цвет. Вы обращаетесь не к одному ботанику, а к трёмстам — и каждый из них смотрит лишь на часть признаков и знает лишь часть примеров. Каждый выдаёт свой ответ, а итог определяется голосованием большинства. Случайные ошибки отдельных «экспертов» при этом взаимно гасятся, и голосование оказывается надёжнее любого одного голоса.

Именно так работает случайный лес: он строит сотни решающих деревьев на случайных подвыборках данных и признаков, а затем усредняет их предсказания.

**Ключевые понятия, которые встретятся в блокноте:**

- **Признак** — один измеренный параметр наблюдения (столбец таблицы), например, длина волны или концентрация вещества.
- **Обучающая выборка** — часть данных, на которой модель учится. **Тестовая выборка** — отложенная часть, которую модель не видела; на ней проверяют качество.
- **Переобучение** — ситуация, когда модель запомнила особенности конкретных обучающих примеров и плохо работает на новых данных.
- **Ансамбль** — набор нескольких моделей, чьи ответы объединяются для получения итогового предсказания.
- **Метрика** — числовая мера качества предсказания (например, доля верных ответов).
- **Гиперпараметр** — настройка алгоритма, задаваемая исследователем до обучения (например, число деревьев `n_estimators`).

## 1. Установка библиотек

В среде Google Colab `scikit-learn`, `numpy`, `pandas` и `matplotlib` уже установлены. Ячейка ниже гарантирует их наличие и при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

В качестве примера используем открытый набор по биохимическим характеристикам вина (`wine` из `scikit-learn`): 178 образцов, 13 числовых признаков (алкоголь, зольность, интенсивность цвета и т. д.), три сорта. Задача — отнести образец к сорту по его измеренным характеристикам.

Это типичная задача **классификации**: на входе — таблица измерений, на выходе — предсказанная метка класса. Ячейка ниже загружает данные и показывает первые строки таблицы.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
X = data.data           # таблица признаков
y = data.target         # известный ответ (сорт)
feature_names = list(X.columns)
class_names = list(data.target_names)

print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
print(f"Классы: {class_names}")
X.head()

## 4. Применение метода

**Шаг 4.1. Разбивка на обучающую и тестовую выборки.**

Прежде чем обучать модель, откладываем 30 % данных в тестовую выборку. Модель увидит их только один раз — при финальной оценке. Это гарантирует честное измерение качества: модель оценивается на примерах, которых она не видела в процессе обучения. Параметр `stratify=y` гарантирует, что пропорции классов в обеих частях одинаковы.

**Шаг 4.2. Обучение случайного леса.**

`RandomForestClassifier` строит `n_estimators=300` деревьев. Каждое дерево обучается на случайной подвыборке наблюдений и признаков. Параметр `random_state` фиксирует генератор случайных чисел для воспроизводимости результата.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# n_estimators - число деревьев в лесу; random_state фиксирует воспроизводимость.
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)
print("Модель обучена на", X_train.shape[0], "наблюдениях.")

**Шаг 4.3. Оценка качества на тестовой выборке.**

Ячейка ниже применяет обученную модель к тестовой выборке и печатает:
- **Долю верных ответов (accuracy)** — общий процент правильных предсказаний.
- **Таблицу precision / recall / F1** — метрики по каждому классу отдельно. `precision` — насколько редко модель ошибочно относит другие классы к данному; `recall` — какую долю объектов данного класса модель нашла правильно.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
print(f"Доля верных ответов на тесте: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, target_names=class_names))

### Шаг 4.4. Матрица ошибок и важность признаков

Ячейка ниже строит два графика:

1. **Матрица ошибок** — таблица, показывающая, как модель путает классы. Строки — истинные классы, столбцы — предсказанные.
2. **Важность признаков** методом перестановок (`permutation importance`): признак поочерёдно «перемешивается» случайным образом, и измеряется, насколько при этом падает качество модели. Чем сильнее падение — тем важнее признак. Этот метод надёжнее встроенных оценок леса, которые смещены в сторону признаков с большим числом уникальных значений.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Матрица ошибок
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, cmap="Blues",
    ax=axes[0], colorbar=False)
axes[0].set_title("Матрица ошибок")
axes[0].set_xlabel("Предсказанный класс")
axes[0].set_ylabel("Истинный класс")
axes[0].grid(False)

# Важность признаков (перестановочная)
perm = permutation_importance(model, X_test, y_test, n_repeats=20,
                              random_state=42)
order = np.argsort(perm.importances_mean)
axes[1].barh(np.array(feature_names)[order], perm.importances_mean[order],
             xerr=perm.importances_std[order], color=VIZ["series"][0])
axes[1].set_title("Важность признаков (перестановочная)")
axes[1].set_xlabel("Среднее падение качества")
axes[1].grid(True, axis="x")

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Матрица ошибок (левый график)**: диагональные клетки — верные ответы (их должно быть больше всего). Числа вне диагонали — ошибки: строка показывает истинный класс, столбец — что предсказала модель. Если почти все ошибки сосредоточены в одной паре строка/столбец, значит именно эти два класса трудноразличимы по имеющимся признакам.

- **Важность признаков (правый график)**: горизонтальная полоска — среднее падение качества при перемешивании данного признака; усы — разброс между повторными перемешиваниями. Признаки отсортированы снизу вверх: самые важные — наверху. Признаки, чья полоска практически равна нулю, можно убрать без потери качества.

## 5. Интерпретация результата

- **Доля верных ответов** — общая точность; для несбалансированных классов смотрите также `precision` и `recall` по каждому классу в отчёте выше.
- **Матрица ошибок**: на диагонали — верные ответы, вне диагонали — перепутанные классы. Сосредоточение ошибок в одной паре классов указывает, какие группы похожи по измерениям.
- **Важность признаков**: чем длиннее столбец, тем сильнее признак влияет на предсказание. Признаки у верха графика — кандидаты на содержательную интерпретацию и потенциально достаточный набор для упрощённой модели.

Важно помнить: случайный лес плохо экстраполирует за пределы диапазона обучающих данных, а сама по себе высокая важность признака не означает причинно-следственной связи.

## 5б. Наглядный эксперимент: как лес «рисует» границу классов

Чтобы увидеть, что делает случайный лес — построим искусственный двумерный пример: три компактных облака точек, каждое своего класса. Модель обучается и мы рисуем её «карту решений» — какому классу она отнесёт любую точку плоскости. Это позволяет увидеть нелинейную, гибкую границу, которую строит ансамбль деревьев.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.ensemble import RandomForestClassifier

# Синтетические данные: три класса на плоскости.
X2, y2 = make_blobs(n_samples=300, centers=[[-3, -2], [2, 3], [4, -2]],
                    cluster_std=1.1, random_state=7)

clf2 = RandomForestClassifier(n_estimators=200, random_state=42)
clf2.fit(X2, y2)

# Сетка точек для отрисовки карты решений.
xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 300),
                     np.linspace(X2[:, 1].min() - 1, X2[:, 1].max() + 1, 300))
Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

palette = get_palette(3)
# Светлые оттенки фона для зон классов.
bg_colors = ["#dbeafe", "#ccfbf1", "#fef3c7"]

fig, ax = plt.subplots(figsize=(7.5, 6))
for cls in range(3):
    ax.contourf(xx, yy, Z == cls, levels=[0.5, 1.5],
                colors=[bg_colors[cls]], alpha=0.55)
for cls in range(3):
    mask = y2 == cls
    ax.scatter(X2[mask, 0], X2[mask, 1], s=35, color=palette[cls],
               edgecolors="white", linewidths=0.5,
               label=f"Класс {cls + 1}")

ax.set_title("Карта решений случайного леса\n(синтетические данные, 2 признака)")
ax.set_xlabel("Признак 1")
ax.set_ylabel("Признак 2")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

**Как читать этот график:**

Закрашенные зоны — области, которые модель относит к тому или иному классу. Точки — обучающие наблюдения. Обратите внимание: граница между зонами не прямая и не круглая — она изломана и адаптируется к форме облаков точек. Именно это отличает ансамбль деревьев от линейной модели: граница может быть произвольно сложной.

Небольшие «острова» чужого цвета внутри зоны — нормальное явление для случайного леса: ансамбль «запомнил» отдельные точки. Если таких островов очень много — возможно переобучение.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу, где строки — наблюдения, столбцы — числовые признаки, и отдельный столбец — известный ответ (метка класса).

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и укажите имя файла и название столбца с ответом.
3. Последовательно выполните ячейки разделов 4 и 5 — код переиспользуется без изменений.

## Поэкспериментируйте сами

Вернитесь к ячейке обучения модели (раздел 4) и попробуйте изменить один параметр за раз — наблюдайте, как меняются результаты:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `n_estimators` | Уменьшите до 10, затем увеличьте до 1000 | Как меняется точность и время обучения? |
| `max_features` | Добавьте `max_features="sqrt"` (по умолчанию) или `max_features=1` | Меняется ли важность признаков? |
| `max_depth` | Добавьте `max_depth=3` | Модель становится проще — как меняется точность? |
| `test_size` | В `train_test_split` измените на `0.1` или `0.5` | Что происходит с качеством при меньшем числе обучающих примеров? |

Также попробуйте убрать из обучения два-три наименее важных признака (по графику важности) и переобучить модель — станет ли она хуже?

## Краткие выводы

- Случайный лес — ансамблевый метод: итоговый ответ определяется голосованием сотен независимых деревьев, каждое из которых видело лишь часть данных. Это делает модель устойчивой и простой в применении.
- Метод хорошо работает «из коробки» на табличных данных с нелинейными зависимостями, не требует масштабирования признаков и устойчив к выбросам.
- Важность признаков (permutation importance) показывает, какие измерения несут наибольшую информацию — это ценно для научной интерпретации.
- Основные ограничения: плохая экстраполяция за пределы обучающего диапазона, высокое потребление памяти при очень большом числе деревьев, меньшая точность по сравнению с настроенным градиентным бустингом.
- Следующий шаг: сравните результат с градиентным бустингом (следующий блокнот проекта) — на тех же данных.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv("my_data.csv")          # ваш файл
# target_column = "класс"                  # имя столбца с ответом
#
# y = df[target_column]
# X = df.drop(columns=[target_column])
# feature_names = list(X.columns)
# class_names = [str(c) for c in sorted(y.unique())]
#
# print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
# Далее повторите ячейки раздела 4.